### Distribución de vitales: primer día frente a último día (v4)

Este notebook muestra **qué cambia en los datos** en los escenarios de prueba de v4:

1. **Covariate drift:** la distribución de cada vital se desplaza día a día. Se
   compara el día 1 (azul, sin cambio) con el día 16 (naranja, 12 días de rampa).
2. **Concept drift:** los vitales **no** cambian; cambia la relación entre vitales
   y etiqueta artificial. Se muestran las mismas distribuciones (que coinciden) y
   la tasa de etiqueta positiva por decil de cada vital, día 1 frente a día 16.

Los CSV de `outputs/` solo guardan métricas agregadas, así que el replay de B-test
se reconstruye con el mismo código del notebook v4 (`../input/main.ipynb`, celdas
de datos, particiones, replay e inyección) y las velocidades calibradas en
`run_summary.json`. No se entrena ningún agente. Requiere `physionet.parquet` en
`data/processed/physionet/` del repositorio o en `/kaggle/input/`.

### Rutas y velocidades calibradas

Se localiza la raíz del repositorio, el notebook v4 y el parquet. Las salidas del
código reutilizado (por ejemplo su `run_summary.json`) se envían a una carpeta
temporal para no sobrescribir los outputs reales de v4.

In [ ]:
import json, os, tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd()
OUTPUTS = next(p for p in (HERE, HERE / 'kaggle/dqn-dummy/v4/outputs') if (p / 'run_summary.json').is_file())
ROOT = next(p for p in (OUTPUTS, *OUTPUTS.parents) if (p / 'data/processed/physionet/physionet.parquet').is_file() or p == Path('/kaggle/working'))
summary = json.loads((OUTPUTS / 'run_summary.json').read_text(encoding='utf-8'))
COV_RATE = summary['cov_rate_iqr_per_day']
CONCEPT_RATE = summary['concept_rate_per_day']
parquet = ROOT / 'data/processed/physionet/physionet.parquet'
if parquet.is_file():
    os.environ['DARL_DUMMY_PARQUET'] = str(parquet)
os.environ.pop('DARL_DUMMY_SMOKE', None)
os.environ['DARL_DUMMY_OUTPUT'] = tempfile.mkdtemp(prefix='darl_v4_drift_')
(OUTPUTS / 'figures').mkdir(exist_ok=True)
print('Outputs v4:', OUTPUTS.resolve())
print(f'Velocidades: covariate {COV_RATE} IQR/día | concept {CONCEPT_RATE}/día')

### Reutilizar el código de datos del notebook v4

Se ejecutan solo las celdas `cell-01-code` a `cell-07-code` de `../input/main.ipynb`:
configuración, carga, particiones y etiqueta, replay, pipeline inicial, inyección
gradual y monitoreo. Tarda alrededor de un minuto.

In [ ]:
source_nb = json.loads((OUTPUTS.parent / 'input' / 'main.ipynb').read_text(encoding='utf-8'))
wanted = [f'cell-0{i}-code' for i in range(1, 8)]
cells = {c['id']: ''.join(c['source']) for c in source_nb['cells'] if c['cell_type'] == 'code'}
cwd = Path.cwd()
os.chdir(ROOT)
try:
    for cell_id in wanted:
        exec(cells[cell_id], globals())
finally:
    os.chdir(cwd)
assert DAYS == 16 and not SMOKE
print('Código v4 cargado; DAYS =', DAYS)

### Replay de prueba con cambio gradual

Mismo replay de B-test que en la evaluación final (semilla 42). Se usan las 4.800
filas de cada día (pacientes de actualización y de evaluación). El día 16 de
covariate tiene un desplazamiento de `12 · 0,25 = 3` IQR; el día 16 de concept
está en la posición `12 · 0,15 = 1,8` de la trayectoria de pesos.

In [ ]:
test_base = make_replay(B_PARTS['test'], days=DAYS, seed=SEED)
FIRST, LAST = 1, DAYS
scenarios = {name: inject(test_base, name, COV_RATE, CONCEPT_RATE) for name in ('covariate', 'concept')}
for name, schedule in scenarios.items():
    last = schedule.loc[schedule.day.eq(LAST)].iloc[0]
    print(f'{name}: día {LAST} cov_level={last.cov_level:.2f} IQR | concept_level={last.concept_level:.2f}')

### Figura 1: covariate drift en los siete vitales

Histogramas de densidad con los mismos intervalos para ambos días. La deriva
desplaza la mediana `L · IQR` en la dirección de cada vital y ensancha la
dispersión con la escala `s(L)`.

In [ ]:
BLUE, ORANGE = '#1f6f9e', '#b3741f'
UNITS = {'HR': 'lpm', 'O2Sat': '%', 'Temp': '°C', 'SBP': 'mmHg', 'MAP': 'mmHg', 'DBP': 'mmHg', 'Resp': 'rpm'}
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})

def day_rows(schedule, day):
    """Return all replay rows of one simulated day."""
    return schedule.loc[schedule.day.eq(day)]

def plot_distributions(schedule, title, filename):
    """Overlay first-day and last-day vital histograms with shared bins."""
    first, last = day_rows(schedule, FIRST), day_rows(schedule, LAST)
    fig, axes = plt.subplots(2, 4, figsize=(16, 9))
    for ax, vital in zip(axes.flat, VITALS):
        a, b = first[vital].dropna(), last[vital].dropna()
        lo, hi = np.nanpercentile(np.r_[a, b], [.5, 99.5])
        bins = np.linspace(lo, hi, 40)
        ax.hist(a, bins=bins, density=True, histtype='stepfilled', alpha=.35, color=BLUE, label=f'Día {FIRST}')
        ax.hist(b, bins=bins, density=True, histtype='stepfilled', alpha=.35, color=ORANGE, label=f'Día {LAST}')
        ax.hist(a, bins=bins, density=True, histtype='step', lw=1.5, color=BLUE)
        ax.hist(b, bins=bins, density=True, histtype='step', lw=1.5, color=ORANGE)
        ax.axvline(a.median(), color=BLUE, ls='--', lw=1)
        ax.axvline(b.median(), color=ORANGE, ls='--', lw=1)
        ks = ks_2samp(a, b, method='asymp').statistic
        ax.set_title(f'{vital}  ·  KS = {ks:.2f}', fontsize=11)
        ax.set_xlabel(UNITS[vital]); ax.set_yticks([])
    ax = axes.flat[-1]
    ax.axis('off')
    handles, labels = axes.flat[0].get_legend_handles_labels()
    ax.legend(handles, labels, loc='center', frameon=False, fontsize=12,
              title='Línea discontinua = mediana')
    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    for ext in ('png', 'svg'):
        fig.savefig(OUTPUTS / 'figures' / f'{filename}.{ext}', dpi=160 if ext == 'png' else None, bbox_inches='tight')
    plt.show()

plot_distributions(scenarios['covariate'],
                   f'Covariate drift: vitales del día {FIRST} frente al día {LAST} (desplazamiento 3 IQR)',
                   '10_distribucion_vitales_covariate')

### Figura 2: concept drift

**Panel superior:** mismas distribuciones que la figura 1 pero en el escenario
concept; ambos días coinciden porque las mediciones no se transforman. El KS
residual (≈0,03–0,06) se debe solo al recambio de pacientes entre días.

**Panel inferior:** tasa de etiqueta positiva `P(y = 1 | decil del vital)`, con
deciles fijados en el día 1. Es donde aparece el concept drift: al invertirse el
peso de HR (y en parte el de Resp), la relación entre esos vitales y la etiqueta
cambia de pendiente, aunque la distribución de los vitales sea la misma.

In [ ]:
def positive_rate_by_decile(rows, vital, edges):
    """Return mean dummy label per fixed decile bin of one vital."""
    values = rows[vital].to_numpy(dtype=float)
    keep = np.isfinite(values)
    idx = np.clip(np.searchsorted(edges, values[keep], side='right') - 1, 0, len(edges) - 2)
    y = rows.y_dummy.to_numpy()[keep]
    return np.array([y[idx == k].mean() if np.any(idx == k) else np.nan for k in range(len(edges) - 1)])

schedule = scenarios['concept']
first, last = day_rows(schedule, FIRST), day_rows(schedule, LAST)
fig, axes = plt.subplots(2, 7, figsize=(16, 9), gridspec_kw={'height_ratios': [1, 1.3]})
for j, vital in enumerate(VITALS):
    a, b = first[vital].dropna(), last[vital].dropna()
    lo, hi = np.nanpercentile(np.r_[a, b], [.5, 99.5])
    bins = np.linspace(lo, hi, 30)
    ax = axes[0, j]
    ax.hist(a, bins=bins, density=True, histtype='stepfilled', alpha=.35, color=BLUE, label=f'Día {FIRST}')
    ax.hist(b, bins=bins, density=True, histtype='step', lw=1.5, color=ORANGE, label=f'Día {LAST}')
    ax.set_title(f'{vital}\nKS = {ks_2samp(a, b, method="asymp").statistic:.2f}', fontsize=10)
    ax.set_yticks([]); ax.tick_params(axis='x', labelsize=8)
    edges = np.unique(np.nanquantile(a, np.linspace(0, 1, 11)))
    centers = np.arange(len(edges) - 1) + 1
    ax = axes[1, j]
    ax.plot(centers, positive_rate_by_decile(first, vital, edges), '-o', ms=4, lw=2, color=BLUE, label=f'Día {FIRST}')
    ax.plot(centers, positive_rate_by_decile(last, vital, edges), '--o', ms=4, lw=2, color=ORANGE, label=f'Día {LAST}')
    ax.set_ylim(0, 1); ax.set_xticks([1, 5, 10]); ax.set_xlabel('Decil (día 1)', fontsize=9)
    ax.grid(axis='y', color='#eceef0', lw=.8)
    if j:
        ax.set_yticklabels([])
axes[0, 0].set_ylabel('Distribución')
axes[1, 0].set_ylabel('P(y = 1 | decil)')
handles, labels = axes[1, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False, bbox_to_anchor=(.5, .985))
fig.suptitle(f'Concept drift: mismos vitales, distinta relación con la etiqueta (día {FIRST} vs día {LAST})',
             fontsize=14, y=1.02)
fig.tight_layout()
for ext in ('png', 'svg'):
    fig.savefig(OUTPUTS / 'figures' / f'11_concept_drift_relacion.{ext}', dpi=160 if ext == 'png' else None, bbox_inches='tight')
plt.show()

### Tabla de soporte

KS por vital entre día 1 y día 16 en cada escenario, prevalencia de la etiqueta y
AUC del pipeline inicial (sin actualizar) sobre los pacientes de evaluación de
cada día. Covariate mueve los vitales (KS ≈ 0,9); concept no los mueve (KS ≈ 0,04,
recambio de pacientes), pero ambos degradan el modelo inicial.

En O2Sat y Resp hay menos de diez deciles porque muchos valores se repiten.
Valores fuera de rango fisiológico (por ejemplo MAP negativa) son efecto de la
transformación sintética de 3 IQR, no mediciones reales.

In [ ]:
rows = []
for name, schedule in scenarios.items():
    first, last = day_rows(schedule, FIRST), day_rows(schedule, LAST)
    row = {'escenario': name}
    row.update({f'KS_{v}': ks_2samp(first[v].dropna(), last[v].dropna(), method='asymp').statistic for v in VITALS})
    row.update({'prevalencia_dia1': first.y_dummy.mean(), 'prevalencia_dia16': last.y_dummy.mean(),
                'AUC_inicial_dia1': auc(first.loc[first.role.eq('eval')], INITIAL),
                'AUC_inicial_dia16': auc(last.loc[last.role.eq('eval')], INITIAL)})
    rows.append(row)
drift_table = pd.DataFrame(rows).set_index('escenario').round(3)
drift_table.to_csv(OUTPUTS / 'drift_dia1_vs_dia16.csv')
drift_table.T